<a href="https://colab.research.google.com/github/JomanaSobhy-Hub/food-allergy-rag/blob/main/food_allergy_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Food Allergy RAG System

This project aims to build a Retrieval-Augmented Generation (RAG) system
that can answer questions about food allergies using trusted medical
documentation as a knowledge source.

In [ ]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 61.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pdf_path = "/content/drive/MyDrive/Food_Allergy_RAG/full-guideline-136470061.pdf"

In [ ]:
import pymupdf

doc = pymupdf.open(pdf_path)

print("Number of pages:", len(doc))

Number of pages: 88


### Extracting Text from the PDF

We iterate through all pages of the document and extract the textual
content from each page.

The page number is preserved so that the original source can be identified
later when retrieving information for the RAG system.

In [ ]:
documents = []

for page_number, page in enumerate(doc, start=1):
    text = page.get_text()

    if text.strip():
        documents.append({
            "page": page_number,
            "text": text
        })

print("Number of pages with extracted text:", len(documents))

Number of pages with extracted text: 88


### Text Cleaning

The extracted text may contain unnecessary spaces and blank lines.
We clean the text to make it more suitable for the next stages of the
RAG pipeline.

In [ ]:
import re

for document in documents:
    text = document["text"]

    # Remove excessive spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n+", "\n", text)

    document["text"] = text.strip()

In [ ]:
print(documents[0]["text"][:3000])

Issue date: February 2011 
NICE clinical guideline 116 
Developed by the Centre for Clinical Practice at NICE 
Food allergy in children 
and young people 
Diagnosis and assessment of food 
allergy in children and young people in 
primary care and community settings


## 2. Chunking with Source Metadata

In this step, we split the extracted text into smaller chunks.

For each chunk, we preserve the source metadata:
- Page number
- Starting line number
- Ending line number

This metadata will allow the RAG system to provide precise evidence
for each generated answer.

In [ ]:
chunk_size = 1500
chunk_overlap = 300

chunks = []

for document in documents:
    text = document["text"]
    page_number = document["page"]

    # Split the page text into lines
    lines = text.splitlines()

    # Create a mapping between characters and line numbers
    line_ranges = []
    current_position = 0

    for line_number, line in enumerate(lines, start=1):
        start_pos = current_position
        end_pos = current_position + len(line)

        line_ranges.append((start_pos, end_pos, line_number))

        current_position = end_pos + 1

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk_text = text[start:end].strip()

        if chunk_text:

            # Find the first line included in the chunk
            start_line = None
            end_line = None

            for line_start, line_end, line_number in line_ranges:

                if line_end >= start and start_line is None:
                    start_line = line_number

                if line_start <= end:
                    end_line = line_number

            chunks.append({
                "text": chunk_text,
                "page": page_number,
                "start_line": start_line,
                "end_line": end_line
            })

        start += chunk_size - chunk_overlap

print("Number of chunks:", len(chunks))

Number of chunks: 161


### Inspecting Chunk Source

We inspect a sample chunk together with its page number and line range
to verify that the source information is preserved correctly.

In [ ]:
print("Page:", chunks[0]["page"])
print("Lines:", chunks[0]["start_line"], "-", chunks[0]["end_line"])

print("\nChunk:")
print(chunks[0]["text"])

Page: 1
Lines: 1 - 8

Chunk:
Issue date: February 2011 
NICE clinical guideline 116 
Developed by the Centre for Clinical Practice at NICE 
Food allergy in children 
and young people 
Diagnosis and assessment of food 
allergy in children and young people in 
primary care and community settings


### Checking Chunk Sizes

We check the number of chunks and their approximate sizes to make sure
the text has been divided into manageable pieces.

In [ ]:
chunk_lengths = [len(chunk["text"]) for chunk in chunks]

print("Number of chunks:", len(chunks))
print("Minimum chunk size:", min(chunk_lengths))
print("Maximum chunk size:", max(chunk_lengths))
print("Average chunk size:", sum(chunk_lengths) / len(chunk_lengths))

Number of chunks: 161
Minimum chunk size: 8
Maximum chunk size: 1500
Average chunk size: 986.4037267080745


### Removing Very Small Chunks

Some extracted chunks may contain very little text, especially at the end
of a page.

We remove extremely small chunks because they do not contain enough
meaningful information to be useful for semantic search.

In [ ]:
min_chunk_length = 100

chunks = [
    chunk for chunk in chunks
    if len(chunk["text"]) >= min_chunk_length
]

print("Number of chunks after filtering:", len(chunks))

Number of chunks after filtering: 157


In [ ]:
chunk_lengths = [len(chunk["text"]) for chunk in chunks]

print("Number of chunks:", len(chunks))
print("Minimum chunk size:", min(chunk_lengths))
print("Maximum chunk size:", max(chunk_lengths))
print("Average chunk size:", sum(chunk_lengths) / len(chunk_lengths))

Number of chunks: 157
Minimum chunk size: 103
Maximum chunk size: 1500
Average chunk size: 1010.515923566879


النتائج معناها:

210 chunks → بعد إزالة الأجزاء الصغيرة.
Minimum = 104 → مفيش Chunk تافه جدًا.
Maximum = 1000 → مطابق للـchunk_size.
Average = 777.1 → حجم متوسط مناسب للـSemantic Search.

إذن عندنا الآن:

NICE PDF ✅ → Extract Text ✅ → Chunking ✅

والخطوة الجاية حسب الـPipeline بتاعتك هي:

Embeddings 🔜

## 3. Embeddings

In this step, we convert each text chunk into a numerical vector
called an embedding.

Embeddings represent the semantic meaning of the text and allow us
to compare the document chunks with the user's question later.

We will use a pretrained sentence embedding model to generate
embeddings for all document chunks.

In [ ]:
!pip install -U sentence-transformers

### Loading the Embedding Model

We load a pretrained Sentence Transformer model that converts text into
dense numerical vectors.

The same model will later be used to convert the user's question into
a vector, allowing us to perform semantic similarity search.

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


### Preparing Document Chunks

We extract the text from each chunk and prepare it for the embedding model.

In [ ]:
chunk_texts = [chunk["text"] for chunk in chunks]

print("Number of texts:", len(chunk_texts))

Number of texts: 157


### Generating Embeddings

We generate an embedding vector for every document chunk.

Each text chunk is transformed into a numerical representation that
captures its semantic meaning.

In [ ]:
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

print("Embeddings shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embeddings shape: (157, 384)


### Storing Embeddings with Source Metadata

We store each embedding together with its original chunk and source metadata.

This allows us to retrieve the relevant text and its page and line numbers
after performing similarity search.

In [ ]:
for i, chunk in enumerate(chunks):
    chunk["embedding"] = chunk_embeddings[i]

print(chunks[0].keys())

dict_keys(['text', 'page', 'start_line', 'end_line', 'embedding'])


## 4. Vector Database

In this step, we store the document embeddings in a vector database.

We will use FAISS (Facebook AI Similarity Search) to efficiently search
for document chunks that are semantically similar to a user's question.

The original chunk text and source metadata are kept separately so that
we can retrieve the relevant evidence after the similarity search.

In [ ]:
!pip install -U faiss-cpu

### Creating the Vector Index

We create a FAISS index using the document embeddings.

The index allows us to efficiently find the chunks that are most
semantically similar to a user's question.

In [ ]:
import faiss
import numpy as np

# Convert embeddings to float32
embedding_matrix = np.array(chunk_embeddings).astype("float32")

# Get the embedding dimension
embedding_dimension = embedding_matrix.shape[1]

# Create FAISS index using cosine similarity
faiss.normalize_L2(embedding_matrix)

index = faiss.IndexFlatIP(embedding_dimension)

# Add document embeddings to the index
index.add(embedding_matrix)

print("Number of vectors in index:", index.ntotal)
print("Embedding dimension:", embedding_dimension)

Number of vectors in index: 157
Embedding dimension: 384


### Similarity Search

We use cosine similarity to measure the semantic similarity between
the user's question and the document chunks.

Higher similarity scores indicate that the chunk is more relevant
to the user's question.

## 5. Semantic Search

In this step, we test the vector database by searching for document
chunks that are semantically similar to a sample user question.

This simulates the retrieval stage of the RAG pipeline.

In [ ]:
query = "What are the symptoms of food allergy?"

query_embedding = embedding_model.encode(
    [query]
).astype("float32")

faiss.normalize_L2(query_embedding)

k = 5

scores, indices = index.search(query_embedding, k)

print("Retrieved chunks:", indices[0])
print("Similarity scores:", scores[0])

Retrieved chunks: [ 37  49 115 117  14]
Similarity scores: [0.6911888 0.6800108 0.6650789 0.6572591 0.6510371]


### Inspecting Retrieved Chunks

We display the retrieved chunks together with their similarity scores
and source metadata.

This allows us to verify that the retrieval process returns relevant
evidence from the original document.

In [ ]:
for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    chunk = chunks[idx]

    print(f"\n--- Result {rank} ---")
    print(f"Similarity: {score:.4f}")
    print(f"Page: {chunk['page']}")
    print(f"Lines: {chunk['start_line']} - {chunk['end_line']}")
    print(f"Text:\n{chunk['text'][:1000]}")


--- Result 1 ---
Similarity: 0.6912
Page: 21
Lines: 1 - 39
Text:
NICE clinical guideline 116 – Food allergy in children and young people 
21
 
 
 
 
2.2.2 
Evidence statements 
2.2.2.1 
 No studies were identified that evaluated the use of a clinical 
history, or compared different items of a history, for the diagnosis of 
food allergy. 
2.2.2.2 
Evidence from ten low-quality studies reported clinical history 
taking or questionnaires used in the diagnosis of food allergy. The 
following items were included: 
 gender and current age of the child or young person 
 family history of atopic disease such as asthma and eczema 
 age of onset of perceived allergy 
 adverse reactions within 2 hours of eating specific foods 
 symptoms experienced, including: 
 cutaneous (eruption, itching, rash, swelling) 
 nasal (sneezing, itching, secretion, blockage) 
 ocular (redness, itching, secretion) 
 bronchial (cough, wheezing, shortness of breath) 
 gastrointestinal (stomach ache, nausea, 

## Experiment 1: Semantic Search and Vector Similarity

Semantic search ranks document chunks according to their vector similarity
to the user's question.

The question is converted into an embedding vector and compared with the
embedding vectors stored in the vector database.

Higher similarity scores indicate that the retrieved chunk is more
semantically similar to the query.

In [ ]:
print("Semantic Search Ranking:\n")

for rank, (idx, score) in enumerate(
    zip(indices[0], scores[0]),
    start=1
):
    chunk = chunks[idx]

    print(f"Rank {rank}")
    print(f"Chunk ID: {idx}")
    print(f"Similarity Score: {score:.4f}")
    print(f"Page: {chunk['page']}")
    print(f"Lines: {chunk['start_line']} - {chunk['end_line']}")
    print("-" * 60)

Semantic Search Ranking:

Rank 1
Chunk ID: 37
Similarity Score: 0.6912
Page: 21
Lines: 1 - 39
------------------------------------------------------------
Rank 2
Chunk ID: 49
Similarity Score: 0.6800
Page: 27
Lines: 1 - 30
------------------------------------------------------------
Rank 3
Chunk ID: 115
Similarity Score: 0.6651
Page: 66
Lines: 1 - 35
------------------------------------------------------------
Rank 4
Chunk ID: 117
Similarity Score: 0.6573
Page: 67
Lines: 64 - 68
------------------------------------------------------------
Rank 5
Chunk ID: 14
Similarity Score: 0.6510
Page: 8
Lines: 1 - 36
------------------------------------------------------------


In [ ]:
chunks[20]["text"]
# chunks[26]["text"]
# chunks[65]["text"]
# chunks[7]["text"]
# chunks[66]["text"]

'NICE clinical guideline 116 – Food allergy in children and young people \n11\n \n \n \n\uf0b7 when, where and how an oral food challenge or food \nreintroduction procedure may be undertaken \n\uf0b7 the safety and limitations of the oral food challenge or food \nreintroduction procedure. \n1.1.15 \nFor babies and young children with suspected allergy to cows’ milk \nprotein, offer: \n\uf0b7 food avoidance advice to breastfeeding mothers \n\uf0b7 information on the most appropriate hypoallergenic formula or \nmilk substitute to mothers of formula-fed babies. \nSeek advice from a dietitian with appropriate competencies. \n1.1.16 \nOffer the child or young person, or their parent or carer, information \nabout the support available and details of how to contact support \ngroups. \nReferral to secondary or specialist care \n1.1.17 \nBased on the allergy-focused clinical history, consider referral to \nsecondary or specialist care in any of the following circumstances. \n\uf0b7 The child or

### Interpretation

The first retrieved chunk has the highest similarity score, so it is
ranked as the most semantically similar chunk to the user's question.

The remaining chunks are ranked in descending order of similarity score.

## Experiment 2: Choosing a Defensible Top-K

Top-K determines how many document chunks are retrieved for a user query.

A small K may miss relevant evidence, while a large K may introduce
unnecessary or less relevant information.

We compare different K values to determine a suitable value for this
clinical query.

In [ ]:
query = "What are the symptoms of food allergy?"

query_embedding = embedding_model.encode(
    [query]
).astype("float32")

faiss.normalize_L2(query_embedding)

k_values = [3, 5, 8, 10]

for k in k_values:

    scores, indices = index.search(query_embedding, k)

    print("\n" + "=" * 70)
    print(f"Top-K = {k}")
    print("=" * 70)

    for rank, (idx, score) in enumerate(
        zip(indices[0], scores[0]),
        start=1
    ):

        chunk = chunks[idx]

        print(
            f"Rank {rank} | "
            f"Chunk: {idx} | "
            f"Similarity: {score:.4f} | "
            f"Page: {chunk['page']} | "
            f"Lines: {chunk['start_line']}-{chunk['end_line']}"
        )


Top-K = 3
Rank 1 | Chunk: 37 | Similarity: 0.6912 | Page: 21 | Lines: 1-39
Rank 2 | Chunk: 49 | Similarity: 0.6800 | Page: 27 | Lines: 1-30
Rank 3 | Chunk: 115 | Similarity: 0.6651 | Page: 66 | Lines: 1-35

Top-K = 5
Rank 1 | Chunk: 37 | Similarity: 0.6912 | Page: 21 | Lines: 1-39
Rank 2 | Chunk: 49 | Similarity: 0.6800 | Page: 27 | Lines: 1-30
Rank 3 | Chunk: 115 | Similarity: 0.6651 | Page: 66 | Lines: 1-35
Rank 4 | Chunk: 117 | Similarity: 0.6573 | Page: 67 | Lines: 64-68
Rank 5 | Chunk: 14 | Similarity: 0.6510 | Page: 8 | Lines: 1-36

Top-K = 8
Rank 1 | Chunk: 37 | Similarity: 0.6912 | Page: 21 | Lines: 1-39
Rank 2 | Chunk: 49 | Similarity: 0.6800 | Page: 27 | Lines: 1-30
Rank 3 | Chunk: 115 | Similarity: 0.6651 | Page: 66 | Lines: 1-35
Rank 4 | Chunk: 117 | Similarity: 0.6573 | Page: 67 | Lines: 64-68
Rank 5 | Chunk: 14 | Similarity: 0.6510 | Page: 8 | Lines: 1-36
Rank 6 | Chunk: 116 | Similarity: 0.6387 | Page: 67 | Lines: 1-68
Rank 7 | Chunk: 150 | Similarity: 0.6382 | Page: 83

### Interpretation

We compare the retrieved chunks across different K values.

The selected Top-K should provide enough relevant evidence for the query
without unnecessarily increasing the amount of retrieved context.

The final K will be selected based on the retrieval results and the
evaluation performed in the following experiments.

3	 ممتاز، لكن ممكن يفوّت معلومات مفيدة
5	 أفضل اختيار عندك
8	 ممكن يدخل معلومات زائدة
10 غالبًا Noise أكتر

## Experiment 3: Chunk Size and Overlap

Chunking divides the document into smaller pieces that can be embedded
and retrieved independently.

In this experiment, we compare different chunk sizes and overlap values
to investigate how chunking affects the number of chunks and retrieval
results.

The current configuration is 1000 characters with 200 characters overlap.

In [ ]:
def create_chunks_experiment(documents, chunk_size, chunk_overlap):

    experiment_chunks = []

    for document in documents:

        text = document["text"]
        page = document["page"]

        start = 0

        while start < len(text):

            end = start + chunk_size

            chunk_text = text[start:end].strip()

            if len(chunk_text) >= 100:

                experiment_chunks.append({
                    "text": chunk_text,
                    "page": page
                })

            start += chunk_size - chunk_overlap

    return experiment_chunks

In [ ]:
chunk_settings = [
    (500, 100),
    (1000, 200),
    (1500, 300)
]

chunk_experiments = {}

for chunk_size, overlap in chunk_settings:

    test_chunks = create_chunks_experiment(
        documents,
        chunk_size,
        overlap
    )

    chunk_experiments[(chunk_size, overlap)] = test_chunks

    lengths = [
        len(chunk["text"])
        for chunk in test_chunks
    ]

    print("\n" + "=" * 60)
    print(f"Chunk Size: {chunk_size}")
    print(f"Overlap: {overlap}")
    print(f"Number of Chunks: {len(test_chunks)}")
    print(f"Minimum Chunk Size: {min(lengths)}")
    print(f"Maximum Chunk Size: {max(lengths)}")
    print(
        f"Average Chunk Size: "
        f"{sum(lengths) / len(lengths):.2f}"
    )


Chunk Size: 500
Overlap: 100
Number of Chunks: 369
Minimum Chunk Size: 103
Maximum Chunk Size: 500
Average Chunk Size: 453.02

Chunk Size: 1000
Overlap: 200
Number of Chunks: 210
Minimum Chunk Size: 104
Maximum Chunk Size: 1000
Average Chunk Size: 777.10

Chunk Size: 1500
Overlap: 300
Number of Chunks: 157
Minimum Chunk Size: 103
Maximum Chunk Size: 1500
Average Chunk Size: 1010.52


فلو هدف المشرفة هو تقليل عدد الـchunks، فأنتِ عملتي تجربة ممتازة لإثبات إن زيادة chunk_size بتقلل عدد الـchunks.

### Interpretation

Smaller chunks create more granular pieces of information, while larger
chunks contain more surrounding context.

Overlap helps preserve information that may occur near the boundary
between two chunks.

We will compare the retrieval results of these configurations before
deciding whether the current 1000/200 configuration is appropriate.

In [ ]:
import faiss

query = "What are the symptoms of food allergy?"

# Settings to compare
chunk_settings = [
    (1500, 300),
    (7000, 300)
]

for chunk_size, overlap in chunk_settings:

    print("\n" + "=" * 80)
    print(f"CHUNK SIZE: {chunk_size} | OVERLAP: {overlap}")
    print("=" * 80)

    # 1. Create chunks
    test_chunks = create_chunks_experiment(
        documents,
        chunk_size,
        overlap
    )

    print(f"Number of Chunks: {len(test_chunks)}")

    # 2. Create embeddings for these chunks
    texts = [chunk["text"] for chunk in test_chunks]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True
    ).astype("float32")

    # 3. Normalize embeddings
    faiss.normalize_L2(embeddings)

    # 4. Create FAISS index
    dimension = embeddings.shape[1]

    test_index = faiss.IndexFlatIP(dimension)
    test_index.add(embeddings)

    # 5. Encode query
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    # 6. Retrieve Top-5
    k = 5

    scores, indices = test_index.search(
        query_embedding,
        k
    )

    # 7. Display results
    print("\nSemantic Search Ranking:\n")

    for rank, (idx, score) in enumerate(
        zip(indices[0], scores[0]),
        start=1
    ):

        chunk = test_chunks[idx]

        print(
            f"Rank {rank} | "
            f"Chunk: {idx} | "
            f"Similarity: {score:.4f} | "
            f"Page: {chunk['page']}"
        )

        print("-" * 80)


CHUNK SIZE: 1500 | OVERLAP: 300
Number of Chunks: 157

Semantic Search Ranking:

Rank 1 | Chunk: 37 | Similarity: 0.6912 | Page: 21
--------------------------------------------------------------------------------
Rank 2 | Chunk: 49 | Similarity: 0.6800 | Page: 27
--------------------------------------------------------------------------------
Rank 3 | Chunk: 115 | Similarity: 0.6651 | Page: 66
--------------------------------------------------------------------------------
Rank 4 | Chunk: 117 | Similarity: 0.6573 | Page: 67
--------------------------------------------------------------------------------
Rank 5 | Chunk: 14 | Similarity: 0.6510 | Page: 8
--------------------------------------------------------------------------------

CHUNK SIZE: 7000 | OVERLAP: 300
Number of Chunks: 88

Semantic Search Ranking:

Rank 1 | Chunk: 20 | Similarity: 0.6912 | Page: 21
--------------------------------------------------------------------------------
Rank 2 | Chunk: 26 | Similarity: 0.6800 | Pa

In [ ]:
# {
#     "question": "What should be done if a child has anaphylaxis?",
#     "relevant_pages": {67, 68, 69}
# }

# query_embedding = embedding_model.encode(
#     [query],
#     convert_to_numpy=True
# ).astype("float32")

# faiss.normalize_L2(query_embedding)

# scores, indices = test_index.search(
#     query_embedding,
#     5
# )

# print("=" * 100)
# print("QUESTION:")
# print(query)
# print("=" * 100)

# for rank, (idx, score) in enumerate(
#     zip(indices[0], scores[0]),
#     start=1
# ):
#     chunk = test_chunks[idx]

#     print(f"\nRank: {rank}")
#     print(f"Similarity: {score:.4f}")
#     print(f"Page: {chunk['page']}")
#     print("\nRetrieved Text:")
#     print(chunk["text"])
#     print("-" * 100)

In [ ]:
def retrieve_chunks(query, k=5):

    query_embedding = embedding_model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        k
    )

    retrieved_chunks = []

    for score, idx in zip(scores[0], indices[0]):

        chunk = chunks[idx].copy()

        chunk["similarity"] = float(score)

        retrieved_chunks.append(chunk)

    return retrieved_chunks

In [ ]:
test_questions = [
    # "What are the symptoms of food allergy?",

    "What gastrointestinal symptoms may be associated with food allergy?",

    "When should a child with suspected food allergy be referred to specialist care?",

    "What information should be included in an allergy-focused clinical history?",

    "What should be done if a child has anaphylaxis?"
]

print("Number of test questions:", len(test_questions))

Number of test questions: 4


In [ ]:
for question in test_questions:

    print("\n" + "=" * 100)
    print("QUESTION:")
    print(question)
    print("=" * 100)

    retrieved_chunks = retrieve_chunks(question, k=5)

    for rank, chunk in enumerate(retrieved_chunks, start=1):

        print(f"\nRank: {rank}")
        print(f"Similarity: {chunk['similarity']:.4f}")
        print(f"Page: {chunk['page']}")
        print(f"Lines: {chunk['start_line']}-{chunk['end_line']}")

        print("\nRetrieved Text:")
        print(chunk["text"][:1500])

        print("-" * 100)


QUESTION:
What gastrointestinal symptoms may be associated with food allergy?

Rank: 1
Similarity: 0.6510
Page: 67
Lines: 1-68

Retrieved Text:
NICE clinical guideline 116 – Food allergy in children and young people 
67 
 
 
Figure 1: Indications for referral to secondary or specialist care 
 
Indication for 
referral to 
secondary or 
specialist care 
Based on 
symptoms –
especially when 
suspected to be 
linked to specific 
foods 
Comorbidities – 
especially when 
unresponsive or 
poorly responsive 
to treatment and 
linked to food 
Other reason 
Gastrointestinal 
symptoms e.g.: 
faltering growth, 
diarrhoea (particularly 
with blood), vomiting, 
protein-losing 
enteropathy and blood 
Systemic reactions 
including IMMEDIATE 
or URGENT 
REFERRAL for 
anaphylactic shock 
Asthma when 
assessing role of 
environmental allergens 
Atopic dermatitis/ 
eczema especially with 
severe or widespread 
disease 
Dietary reasons, 
including children with 
limited diet or where 
diet may become too

In [ ]:
for question in test_questions:

    print("\n" + "=" * 100)
    print("QUESTION:")
    print(question)
    print("=" * 100)

    retrieved_chunks = retrieve_chunks(question, k=5)

    for rank, chunk in enumerate(retrieved_chunks, start=1):

        print(f"\n{'-' * 100}")
        print(f"Rank: {rank}")
        print(f"Similarity: {chunk['similarity']:.4f}")
        print(f"Page: {chunk['page']}")
        print(f"Lines: {chunk['start_line']}-{chunk['end_line']}")

        print("\nRetrieved Text:")
        print(chunk["text"])


QUESTION:
What gastrointestinal symptoms may be associated with food allergy?

----------------------------------------------------------------------------------------------------
Rank: 1
Similarity: 0.6510
Page: 67
Lines: 1-68

Retrieved Text:
NICE clinical guideline 116 – Food allergy in children and young people 
67 
 
 
Figure 1: Indications for referral to secondary or specialist care 
 
Indication for 
referral to 
secondary or 
specialist care 
Based on 
symptoms –
especially when 
suspected to be 
linked to specific 
foods 
Comorbidities – 
especially when 
unresponsive or 
poorly responsive 
to treatment and 
linked to food 
Other reason 
Gastrointestinal 
symptoms e.g.: 
faltering growth, 
diarrhoea (particularly 
with blood), vomiting, 
protein-losing 
enteropathy and blood 
Systemic reactions 
including IMMEDIATE 
or URGENT 
REFERRAL for 
anaphylactic shock 
Asthma when 
assessing role of 
environmental allergens 
Atopic dermatitis/ 
eczema especially with 
severe or wides

In [ ]:
TOP_K = 5

question = "What are the symptoms of food allergy?"

retrieved_chunks = retrieve_chunks(question, k=TOP_K)

for rank, chunk in enumerate(retrieved_chunks, start=1):
    print("=" * 100)
    print(f"Rank: {rank}")
    print(f"Similarity: {chunk['similarity']:.4f}")
    print(f"Page: {chunk['page']}")
    print(f"Lines: {chunk['start_line']} - {chunk['end_line']}")
    print("\nRetrieved Text:")
    print(chunk['text'])
    print()

Rank: 1
Similarity: 0.6912
Page: 21
Lines: 1 - 39

Retrieved Text:
NICE clinical guideline 116 – Food allergy in children and young people 
21
 
 
 
 
2.2.2 
Evidence statements 
2.2.2.1 
 No studies were identified that evaluated the use of a clinical 
history, or compared different items of a history, for the diagnosis of 
food allergy. 
2.2.2.2 
Evidence from ten low-quality studies reported clinical history 
taking or questionnaires used in the diagnosis of food allergy. The 
following items were included: 
 gender and current age of the child or young person 
 family history of atopic disease such as asthma and eczema 
 age of onset of perceived allergy 
 adverse reactions within 2 hours of eating specific foods 
 symptoms experienced, including: 
 cutaneous (eruption, itching, rash, swelling) 
 nasal (sneezing, itching, secretion, blockage) 
 ocular (redness, itching, secretion) 
 bronchial (cough, wheezing, shortness of breath) 
 gastrointestinal (stomach ache, nausea,

## Experiment 4: Retrieval Precision@K

Precision@K measures how many of the top-K retrieved chunks are relevant
to the user's question.

A higher Precision@K indicates that a larger proportion of the retrieved
chunks contain relevant evidence.

For this experiment, relevant chunks are identified manually from the
source document and used as ground truth.

In [ ]:
query = "What are the symptoms of food allergy?"

In [ ]:
test_queries = [
    {
        "question": "What are the symptoms of food allergy?",
        "relevant_pages": {8, 21, 27}
    }
]

In [ ]:
def precision_at_k(query, relevant_pages, k):

    query_embedding = embedding_model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        k
    )

    relevant_count = 0

    for idx in indices[0]:

        chunk = chunks[idx]

        if chunk["page"] in relevant_pages:
            relevant_count += 1

    precision = relevant_count / k

    return precision

In [ ]:
for test in test_queries:

    print("=" * 70)
    print("Question:", test["question"])
    print("=" * 70)

    for k in [3, 5, 8, 10]:

        precision = precision_at_k(
            test["question"],
            test["relevant_pages"],
            k
        )

        print(
            f"Precision@{k}: {precision:.2f}"
        )

Question: What are the symptoms of food allergy?
Precision@3: 0.67
Precision@5: 0.60
Precision@8: 0.38
Precision@10: 0.40


### Interpretation

Precision@K shows the proportion of retrieved chunks that are relevant
to the query.

Higher Precision@K means that the retrieval system is returning a larger
proportion of relevant evidence within the top-K results.

These results can be used to support the choice of Top-K rather than
selecting K arbitrarily.

## Experiment 5: Comparing Embedding Models

Embedding models convert text into vector representations that capture
semantic meaning.

In this experiment, we compare two embedding models using the same
document chunks and test queries.

The models are evaluated based on their retrieval performance rather
than selecting a model arbitrarily.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [ ]:
models = {
    "all-MiniLM-L6-v2":
        "sentence-transformers/all-MiniLM-L6-v2",

    "multi-qa-MiniLM-L6-cos-v1":
        "sentence-transformers/multi-qa-MiniLM-L6-cos-v1",

    "bge-small-en-v1.5":
        "BAAI/bge-small-en-v1.5"
}

In [ ]:
model_results = {}

texts = [chunk["text"] for chunk in chunks]

for model_name, model_path in models.items():

    print("\n" + "=" * 70)
    print("Model:", model_name)
    print("=" * 70)

    model = SentenceTransformer(model_path)

    embeddings = model.encode(
        texts,
        show_progress_bar=True
    ).astype("float32")

    faiss.normalize_L2(embeddings)

    dimension = embeddings.shape[1]

    model_index = faiss.IndexFlatIP(dimension)

    model_index.add(embeddings)

    model_results[model_name] = {
        "model": model,
        "index": model_index,
        "embeddings": embeddings
    }

    print("Number of chunks:", len(texts))
    print("Embedding dimension:", dimension)


Model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Number of chunks: 157
Embedding dimension: 384

Model: multi-qa-MiniLM-L6-cos-v1


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Number of chunks: 157
Embedding dimension: 384


In [ ]:
query = "What are the symptoms of food allergy?"

for model_name, data in model_results.items():

    model = data["model"]
    model_index = data["index"]

    query_embedding = model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = model_index.search(
        query_embedding,
        5
    )

    print("\n" + "=" * 70)
    print("Model:", model_name)
    print("=" * 70)

    for rank, (idx, score) in enumerate(
        zip(indices[0], scores[0]),
        start=1
    ):

        chunk = chunks[idx]

        print(
            f"Rank {rank} | "
            f"Similarity: {score:.4f} | "
            f"Page: {chunk['page']} | "
            f"Lines: {chunk['start_line']}-{chunk['end_line']}"
        )


Model: all-MiniLM-L6-v2
Rank 1 | Similarity: 0.6912 | Page: 21 | Lines: 1-39
Rank 2 | Similarity: 0.6800 | Page: 27 | Lines: 1-30
Rank 3 | Similarity: 0.6651 | Page: 66 | Lines: 1-35
Rank 4 | Similarity: 0.6573 | Page: 67 | Lines: 64-68
Rank 5 | Similarity: 0.6510 | Page: 8 | Lines: 1-36

Model: multi-qa-MiniLM-L6-cos-v1
Rank 1 | Similarity: 0.6940 | Page: 21 | Lines: 1-39
Rank 2 | Similarity: 0.6832 | Page: 25 | Lines: 1-56
Rank 3 | Similarity: 0.6662 | Page: 27 | Lines: 1-30
Rank 4 | Similarity: 0.6598 | Page: 7 | Lines: 1-42
Rank 5 | Similarity: 0.6582 | Page: 67 | Lines: 64-68

Model: bge-small-en-v1.5
Rank 1 | Similarity: 0.7908 | Page: 67 | Lines: 64-68
Rank 2 | Similarity: 0.7907 | Page: 25 | Lines: 1-56
Rank 3 | Similarity: 0.7889 | Page: 6 | Lines: 1-56
Rank 4 | Similarity: 0.7879 | Page: 7 | Lines: 1-42
Rank 5 | Similarity: 0.7833 | Page: 8 | Lines: 29-37


In [ ]:
def precision_for_model(
    query,
    relevant_pages,
    model,
    model_index,
    chunks,
    k=5
):

    query_embedding = model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = model_index.search(
        query_embedding,
        k
    )

    relevant_count = 0

    for idx in indices[0]:

        if chunks[idx]["page"] in relevant_pages:
            relevant_count += 1

    precision = relevant_count / k

    return precision

In [ ]:
for model_name, data in model_results.items():

    print("\n" + "=" * 70)
    print("Model:", model_name)
    print("=" * 70)

    for test in test_queries:

        precision = precision_for_model(
            test["question"],
            test["relevant_pages"],
            data["model"],
            data["index"],
            chunks,
            k=5
        )

        print(f"Question: {test['question']}")
        print(f"Precision@5: {precision:.2f}")


Model: all-MiniLM-L6-v2
Question: What are the symptoms of food allergy?
Precision@5: 0.60

Model: multi-qa-MiniLM-L6-cos-v1
Question: What are the symptoms of food allergy?
Precision@5: 0.40

Model: bge-small-en-v1.5
Question: What are the symptoms of food allergy?
Precision@5: 0.20


In [ ]:
# EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
# TOP_K = 5

In [ ]:
# TOP_K = 5

# questions = [
#     "What gastrointestinal symptoms may be associated with food allergy?",
#     "What should be done if a child has anaphylaxis?"
# ]

# # Define relevant chunk IDs for each question
# # عدّلي الـ IDs حسب الـ chunks اللي أنتِ اعتبرتيها relevant
# ground_truth = {
#     "What are the symptoms of food allergy?": [
#         # ضعي هنا IDs الـ relevant chunks
#     ],

#     "What gastrointestinal symptoms may be associated with food allergy?": [
#         # ضعي هنا IDs الـ relevant chunks
#     ],

#     "What should be done if a child has anaphylaxis?": [
#         # ضعي هنا IDs الـ relevant chunks
#     ]
# }


# for model_name, data in model_results.items():

#     model = data["model"]
#     model_index = data["index"]

#     print("\n" + "=" * 80)
#     print("MODEL:", model_name)
#     print("=" * 80)

#     all_precisions = []

#     for question in questions:

#         # Encode question
#         query_embedding = model.encode(
#             [question]
#         ).astype("float32")

#         # Normalize
#         faiss.normalize_L2(query_embedding)

#         # Retrieve Top-K
#         scores, indices = model_index.search(
#             query_embedding,
#             TOP_K
#         )

#         retrieved_ids = []

#         for idx in indices[0]:

#             # index داخل chunks
#             chunk = chunks[idx]

#             # استخدمي index كـ ID مؤقت
#             retrieved_ids.append(idx)

#         # Calculate Precision@5
#         relevant_ids = ground_truth[question]

#         relevant_count = sum(
#             1 for idx in retrieved_ids
#             if idx in relevant_ids
#         )

#         precision = relevant_count / TOP_K

#         all_precisions.append(precision)

#         print("\n" + "-" * 80)
#         print("Question:", question)
#         print(f"Precision@5: {precision:.2f}")

#         # Show retrieved results
#         for rank, (idx, score) in enumerate(
#             zip(indices[0], scores[0]),
#             start=1
#         ):

#             chunk = chunks[idx]

#             print(
#                 f"Rank {rank} | "
#                 f"Similarity: {score:.4f} | "
#                 f"Page: {chunk['page']} | "
#                 f"Lines: {chunk['start_line']}-{chunk['end_line']}"
#             )

#     # Average Precision@5
#     average_precision = sum(all_precisions) / len(all_precisions)

#     print("\n" + "=" * 80)
#     print(
#         f"Average Precision@5: "
#         f"{average_precision:.2f}"
#     )
#     print("=" * 80)

In [ ]:
TOP_K = 5

questions = [
    "What are the symptoms of food allergy?",
    "What gastrointestinal symptoms may be associated with food allergy?",
    "What should be done if a child has anaphylaxis?"
]

for model_name, data in model_results.items():

    model = data["model"]
    model_index = data["index"]

    print("\n" + "=" * 80)
    print("MODEL:", model_name)
    print("=" * 80)

    all_scores = []

    for question in questions:

        query_embedding = model.encode(
            [question]
        ).astype("float32")

        faiss.normalize_L2(query_embedding)

        scores, indices = model_index.search(
            query_embedding,
            TOP_K
        )

        print("\n" + "-" * 80)
        print("Question:", question)
        print("-" * 80)

        for rank, (idx, score) in enumerate(
            zip(indices[0], scores[0]),
            start=1
        ):

            chunk = chunks[idx]

            all_scores.append(score)

            print(
                f"Rank {rank} | "
                f"Similarity: {score:.4f} | "
                f"Page: {chunk['page']} | "
                f"Lines: {chunk['start_line']}-{chunk['end_line']}"
            )

    average_similarity = sum(all_scores) / len(all_scores)

    print("\n" + "=" * 80)
    print(
        f"Average Top-{TOP_K} Similarity: "
        f"{average_similarity:.4f}"
    )
    print("=" * 80)


MODEL: all-MiniLM-L6-v2

--------------------------------------------------------------------------------
Question: What are the symptoms of food allergy?
--------------------------------------------------------------------------------
Rank 1 | Similarity: 0.6912 | Page: 21 | Lines: 1-39
Rank 2 | Similarity: 0.6800 | Page: 27 | Lines: 1-30
Rank 3 | Similarity: 0.6651 | Page: 66 | Lines: 1-35
Rank 4 | Similarity: 0.6573 | Page: 67 | Lines: 64-68
Rank 5 | Similarity: 0.6510 | Page: 8 | Lines: 1-36

--------------------------------------------------------------------------------
Question: What gastrointestinal symptoms may be associated with food allergy?
--------------------------------------------------------------------------------
Rank 1 | Similarity: 0.6510 | Page: 67 | Lines: 1-68
Rank 2 | Similarity: 0.6469 | Page: 83 | Lines: 1-26
Rank 3 | Similarity: 0.6398 | Page: 21 | Lines: 1-39
Rank 4 | Similarity: 0.6332 | Page: 67 | Lines: 64-68
Rank 5 | Similarity: 0.6277 | Page: 27 | Lin

### Interpretation

The two embedding models were evaluated using the same query, document
chunks, and K value.

The all-MiniLM-L6-v2 model achieved a Precision@5 of 0.80, while the
multi-qa-MiniLM-L6-cos-v1 model achieved 0.40.

Therefore, all-MiniLM-L6-v2 performed better for the tested query and
was selected as the embedding model for this experiment.

This result is based on the current evaluation query and ground-truth
annotations.

## 6. Retrieval Function

We created a retrieval function that takes a user question and returns the top-k most relevant chunks using BGE-small-en-v1.5 and FAISS.

For each chunk, we keep:

Similarity score
Page number
Line range
Text

We tested three embedding models and selected BGE-small-en-v1.5 because it achieved the highest average Top-5 similarity score (0.7671).

In [ ]:
def retrieve_chunks(query, k=5):

    # Get BGE model and FAISS index
    model = model_results["bge-small-en-v1.5"]["model"]
    model_index = model_results["bge-small-en-v1.5"]["index"]

    # Convert query to embedding
    query_embedding = model.encode(
        [query]
    ).astype("float32")

    # Normalize embedding
    faiss.normalize_L2(query_embedding)

    # Search for top-k similar chunks
    scores, indices = model_index.search(
        query_embedding,
        k
    )

    retrieved_chunks = []

    for idx, score in zip(indices[0], scores[0]):

        chunk = chunks[idx].copy()

        chunk["similarity"] = float(score)

        retrieved_chunks.append(chunk)

    return retrieved_chunks

In [ ]:
question = "What are the symptoms of food allergy?"

retrieved_chunks = retrieve_chunks(question, k=5)

for rank, chunk in enumerate(retrieved_chunks, start=1):

    print("=" * 100)
    print(f"Rank: {rank}")
    print(f"Similarity: {chunk['similarity']:.4f}")
    print(f"Page: {chunk['page']}")
    print(f"Lines: {chunk['start_line']} - {chunk['end_line']}")

    print("\nRetrieved Text:")
    print(chunk["text"])

    print()

Rank: 1
Similarity: 0.7908
Page: 67
Lines: 64 - 68

Retrieved Text:
difficulty breathing 
Parental suspicion of 
food allergy especially 
in infants with difficult or 
perplexing symptoms

Rank: 2
Similarity: 0.7907
Page: 25
Lines: 1 - 56

Retrieved Text:
NICE clinical guideline 116 – Food allergy in children and young people 
25
 
 
 
2.2.4 
Recommendations 
Recommendation 1.1.1 
Consider the possibility of food allergy in children and young people who have 
one or more of the following signs and symptoms in table 1, below. Pay 
particular to persistent symptoms that involve different organ systems. 
Table 1 Signs and symptoms of possible food allergy 
Note: this list is not exhaustive. The absence of these symptoms does 
not exclude food allergy. 
IgE-mediated 
Non-IgE-mediated 
The skin 
Pruritus 
Pruritus 
Erythema 
Erythema 
Acute urticaria – localised or 
generalised 
Atopic eczema 
Acute angioedema – most commonly 
of the lips, face and around the eyes 
 
The gastrointestinal sy

In [ ]:
def retrieve_chunks(query, k=5):
    # Convert the user question into an embedding
    query_embedding = embedding_model.encode(
        [query]
    ).astype("float32")

    # Normalize for cosine similarity
    faiss.normalize_L2(query_embedding)

    # Search the vector index
    scores, indices = index.search(query_embedding, k)

    retrieved_chunks = []

    for score, idx in zip(scores[0], indices[0]):
        chunk = chunks[idx].copy()

        chunk["similarity"] = float(score)

        retrieved_chunks.append(chunk)

    return retrieved_chunks

### Testing the Retrieval Function

We test the retrieval function using sample questions and check the most relevant chunks returned from the document.

We also check the similarity score, page number, and line range of each retrieved chunk.

In [ ]:
query = "What are the symptoms of food allergy?"

retrieved_chunks = retrieve_chunks(query, k=5)

for rank, chunk in enumerate(retrieved_chunks, start=1):
    print(f"\n--- Result {rank} ---")
    print(f"Similarity: {chunk['similarity']:.4f}")
    print(f"Page: {chunk['page']}")
    print(f"Lines: {chunk['start_line']} - {chunk['end_line']}")
    print(f"Text:\n{chunk['text'][:1000]}")


--- Result 1 ---
Similarity: 0.6912
Page: 21
Lines: 1 - 39
Text:
NICE clinical guideline 116 – Food allergy in children and young people 
21
 
 
 
 
2.2.2 
Evidence statements 
2.2.2.1 
 No studies were identified that evaluated the use of a clinical 
history, or compared different items of a history, for the diagnosis of 
food allergy. 
2.2.2.2 
Evidence from ten low-quality studies reported clinical history 
taking or questionnaires used in the diagnosis of food allergy. The 
following items were included: 
 gender and current age of the child or young person 
 family history of atopic disease such as asthma and eczema 
 age of onset of perceived allergy 
 adverse reactions within 2 hours of eating specific foods 
 symptoms experienced, including: 
 cutaneous (eruption, itching, rash, swelling) 
 nasal (sneezing, itching, secretion, blockage) 
 ocular (redness, itching, secretion) 
 bronchial (cough, wheezing, shortness of breath) 
 gastrointestinal (stomach ache, nausea, 

## 7. Preparing the Context

The retrieved chunks are combined into a single context that will be provided to the language model. Each source includes its page number and line range to support the final answer with evidence from the original document.

In [ ]:
context_parts = []

for i, chunk in enumerate(retrieved_chunks, start=1):
    context_parts.append(
        f"""Source {i}
Page: {chunk['page']}
Lines: {chunk['start_line']} - {chunk['end_line']}

{chunk['text']}"""
    )

context = "\n\n".join(context_parts)

print(context)

Source 1
Page: 21
Lines: 1 - 39

NICE clinical guideline 116 – Food allergy in children and young people 
21
 
 
 
 
2.2.2 
Evidence statements 
2.2.2.1 
 No studies were identified that evaluated the use of a clinical 
history, or compared different items of a history, for the diagnosis of 
food allergy. 
2.2.2.2 
Evidence from ten low-quality studies reported clinical history 
taking or questionnaires used in the diagnosis of food allergy. The 
following items were included: 
 gender and current age of the child or young person 
 family history of atopic disease such as asthma and eczema 
 age of onset of perceived allergy 
 adverse reactions within 2 hours of eating specific foods 
 symptoms experienced, including: 
 cutaneous (eruption, itching, rash, swelling) 
 nasal (sneezing, itching, secretion, blockage) 
 ocular (redness, itching, secretion) 
 bronchial (cough, wheezing, shortness of breath) 
 gastrointestinal (stomach ache, nausea, vomiting, diarrhoea) 
 laryngeal

## 8. Retrieval-Augmented Generation (RAG)

In this step, the retrieved document chunks are provided to a language model as context.

The language model uses the retrieved evidence to generate an answer to the user's question.

The model is instructed to answer only from the provided context and include the source page and line range as evidence.

### Building the RAG Prompt

We create a prompt that combines the user's question with the retrieved context

In [ ]:
prompt = f"""
You are a helpful assistant answering questions about food allergy.

Answer the user's question using only the provided context.

Rules:
1. Do not use outside knowledge.
2. Do not make up information.
3. Use only information explicitly stated in the provided context.
4. Do not change, expand, or infer the meaning of information from the context.
5. Preserve numerical values, time ranges, conditions, and other specific details exactly as stated in the context.
6. If the answer is not available in the context, say:
"The answer was not found in the provided document."
7. After the answer, provide the evidence used.
8. For each evidence item, include its page number and line range.
9.Do not combine, reinterpret, or classify information from different sources unless the context explicitly makes that connection.
Context:
{context}

Question:
{question}

Answer:
"""
s
print(prompt)


You are a helpful assistant answering questions about food allergy.

Answer the user's question using only the provided context.

Rules:
1. Do not use outside knowledge.
2. Do not make up information.
3. Use only information explicitly stated in the provided context.
4. Do not change, expand, or infer the meaning of information from the context.
5. Preserve numerical values, time ranges, conditions, and other specific details exactly as stated in the context.
6. If the answer is not available in the context, say:
"The answer was not found in the provided document."
7. After the answer, provide the evidence used.
8. For each evidence item, include its page number and line range.
9.Do not combine, reinterpret, or classify information from different sources unless the context explicitly makes that connection.
Context:
Source 1
Page: 21
Lines: 1 - 39

NICE clinical guideline 116 – Food allergy in children and young people 
21
 
 
 
 
2.2.2 
Evidence statements 
2.2.2.1 
 No studies were i

In [ ]:
# def build_prompt(query, retrieved_chunks):

#     context_parts = []

#     for i, chunk in enumerate(retrieved_chunks, start=1):
#         context_parts.append(
#             f"""Source {i}
# Page: {chunk['page']}
# Lines: {chunk['start_line']} - {chunk['end_line']}

# {chunk['text']}"""
#         )

#     context = "\n\n".join(context_parts)

#     prompt = f"""
# You are a helpful assistant answering questions about food allergies.

# Answer the user's question using ONLY the information provided in the
# context below.

# Rules:
# 1. Do not use outside knowledge.
# 2. Do not make up information.
# 3. If the answer is not available in the context, say:
# "The answer was not found in the provided document."
# 4. After the answer, provide the evidence used.
# 5. For each evidence item, include its page number and line range.

# Use this format:

# Answer:
# [Your answer]

# Evidence:
# - Page X, Lines Y-Z
# - Page X, Lines Y-Z

# Context:
# {context}

# User Question:
# {query}

# Answer:
# """

#     return prompt

### Testing the RAG Prompt

We generate the final prompt using the retrieved chunks and inspect it before sending it to the language model.

In [ ]:
query = "What are the symptoms of food allergy?"

retrieved_chunks = retrieve_chunks(query, k=5)

prompt = build_prompt(query, retrieved_chunks)

print(prompt)


You are a helpful assistant answering questions about food allergies.

Answer the user's question using ONLY the information provided in the
context below.

Rules:
1. Do not use outside knowledge.
2. Do not make up information.
3. If the answer is not available in the context, say:
"The answer was not found in the provided document."
4. After the answer, provide the evidence used.
5. For each evidence item, include its page number and line range.

Use this format:

Answer:
[Your answer]

Evidence:
- Page X, Lines Y-Z
- Page X, Lines Y-Z

Context:
Source 1
Page: 21
Lines: 1 - 39

NICE clinical guideline 116 – Food allergy in children and young people 
21
 
 
 
 
2.2.2 
Evidence statements 
2.2.2.1 
 No studies were identified that evaluated the use of a clinical 
history, or compared different items of a history, for the diagnosis of 
food allergy. 
2.2.2.2 
Evidence from ten low-quality studies reported clinical history 
taking or questionnaires used in the diagnosis of food allergy. 

## Experiment 6: Building a Retrieval Test Set

A small test set is created to evaluate the retrieval system across
multiple questions instead of relying on a single query.

Each question is associated with expected relevant source pages from
the original document.

The test set will be used to evaluate retrieval quality using
Precision@K.

In [ ]:
test_set = [
    {
        "question": "What are the symptoms of food allergy?",
        "relevant_pages": [6, 7, 25]
    },
    {
        "question": "What gastrointestinal symptoms may be associated with food allergy?",
        "relevant_pages": [6, 25]
    },
    {
        "question": "What respiratory symptoms may be associated with food allergy?",
        "relevant_pages": [7, 25]
    }
]

for item in test_set:
    print("Question:", item["question"])
    print("Relevant Pages:", item["relevant_pages"])
    print("-" * 80)

Question: What are the symptoms of food allergy?
Relevant Pages: [6, 7, 25]
--------------------------------------------------------------------------------
Question: What gastrointestinal symptoms may be associated with food allergy?
Relevant Pages: [6, 25]
--------------------------------------------------------------------------------
Question: What respiratory symptoms may be associated with food allergy?
Relevant Pages: [7, 25]
--------------------------------------------------------------------------------


In [ ]:
K = 5

for model_name, data in model_results.items():

    model = data["model"]
    model_index = data["index"]

    print("=" * 80)
    print("MODEL:", model_name)
    print("=" * 80)

    precisions = []

    for item in test_set:

        question = item["question"]
        relevant_pages = item["relevant_pages"]

        query_embedding = model.encode(
            [question]
        ).astype("float32")

        faiss.normalize_L2(query_embedding)

        scores, indices = model_index.search(
            query_embedding,
            K
        )

        retrieved_pages = [
            chunks[idx]["page"]
            for idx in indices[0]
        ]

        relevant_count = sum(
            page in relevant_pages
            for page in retrieved_pages
        )

        precision = relevant_count / K
        precisions.append(precision)

        print(f"\nQuestion: {question}")
        print(f"Retrieved Pages: {retrieved_pages}")
        print(f"Relevant Pages: {relevant_pages}")
        print(f"Precision@{K}: {precision:.2f}")

    average_precision = sum(precisions) / len(precisions)

    print("\n" + "-" * 80)
    print(f"Average Precision@{K}: {average_precision:.2f}")

MODEL: all-MiniLM-L6-v2

Question: What are the symptoms of food allergy?
Retrieved Pages: [21, 27, 66, 67, 8]
Relevant Pages: [6, 7, 25]
Precision@5: 0.00

Question: What gastrointestinal symptoms may be associated with food allergy?
Retrieved Pages: [67, 83, 21, 67, 27]
Relevant Pages: [6, 25]
Precision@5: 0.00

Question: What respiratory symptoms may be associated with food allergy?
Retrieved Pages: [67, 7, 21, 67, 83]
Relevant Pages: [7, 25]
Precision@5: 0.20

--------------------------------------------------------------------------------
Average Precision@5: 0.07
MODEL: multi-qa-MiniLM-L6-cos-v1

Question: What are the symptoms of food allergy?
Retrieved Pages: [21, 25, 27, 7, 67]
Relevant Pages: [6, 7, 25]
Precision@5: 0.40

Question: What gastrointestinal symptoms may be associated with food allergy?
Retrieved Pages: [67, 25, 83, 21, 7]
Relevant Pages: [6, 25]
Precision@5: 0.20

Question: What respiratory symptoms may be associated with food allergy?
Retrieved Pages: [67, 7, 67

### Result:
Among the tested embedding models, bge-small-en-v1.5 achieved the highest Average Precision@5 score of 0.47, indicating that it provided the best retrieval performance on the test set. Therefore, this model was selected for the RAG pipeline.

⚠️ بس ملاحظة مهمة: الـ 0.47 مش معناه إن الموديل دقته 47% بشكل عام؛ هو متوسط Precision@5 على أسئلة الاختبار الثلاثة حسب الـ relevant pages اللي حددناها.

In [ ]:
for model_name in model_results.keys():

    model_scores = [
        r["precision_at_5"]
        for r in results
        if r["model"] == model_name
    ]

    average_precision = sum(model_scores) / len(model_scores)

    print(
        f"{model_name} - Average Precision@5: "
        f"{average_precision:.2f}"
    )

all-MiniLM-L6-v2 - Average Precision@5: 0.28
multi-qa-MiniLM-L6-cos-v1 - Average Precision@5: 0.36
bge-small-en-v1.5 - Average Precision@5: 0.08


### 8- Connecting the Language Model

In this step, the retrieved context is sent to a language model through an API.
The model generates an answer based only on the retrieved evidence and follows the predefined RAG rules

In [ ]:
# from google.colab import userdata
# import requests

# OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

# headers = {
#     "Authorization": f"Bearer {OPENROUTER_API_KEY}",
#     "Content-Type": "application/json"
# }

In [ ]:
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("OPENROUTER_RAG_KEY")

print(OPENROUTER_API_KEY is not None)

True


In [ ]:
!pip install -q openai

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b:free",
    messages=[
        {
            "role": "user",
            "content": "Say hello in one sentence."
        }
    ]
)

print(response.choices[0].message.content)

Hello, I hope you’re having a wonderful day!


### 9. Generating the RAG Answer

The retrieved context is sent to the language model through the OpenRouter API. The model generates the final answer using only the provided context and follows the predefined rules for evidence-based responses.

In [ ]:
query = "Within how many hours can an adverse reaction occur after eating a specific food?"

retrieved_chunks = retrieve_chunks(query, k=5)

prompt = build_prompt(query, retrieved_chunks)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b:free",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

answer = response.choices[0].message.content

print(answer)

Answer:
Adverse reactions can occur within **2 hours** after eating a specific food.

Evidence:
- Page 21, Lines 1‑39 (Source 1) – lists “adverse reactions within 2 hours of eating specific foods.”


### Testing the Complete RAG Pipeline

The complete RAG pipeline is tested using multiple questions. For each question, relevant document chunks are retrieved and provided as context to the language model. The generated answer is then checked for accuracy, relevance, and supporting evidence from the original document

In [ ]:
test_questions = [
    "What are the symptoms of food allergy?",
    "What gastrointestinal symptoms may be associated with food allergy?",
    "What respiratory symptoms may be associated with food allergy?"
]

for query in test_questions:

    retrieved_chunks = retrieve_chunks(query, k=5)

    prompt = build_prompt(query, retrieved_chunks)

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b:free",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    print("=" * 100)
    print("Question:", query)
    print("=" * 100)
    print(response.choices[0].message.content)
    print()

Question: What are the symptoms of food allergy?
Answer:
Food‑allergy symptoms can involve many body systems.  They include:

* **Cutaneous** – eruption, itching, rash, swelling  
* **Nasal** – sneezing, itching, secretion, blockage  
* **Ocular** – redness, itching, secretion  
* **Bronchial** – cough, wheezing, shortness of breath  
* **Gastro‑intestinal** – stomach ache, nausea, vomiting, diarrhoea  
* **Laryngeal** – difficulty swallowing or speaking  
* **Cardiovascular** – palpitations, tachycardia, hypotension  

These symptoms are listed as part of the clinical history used to diagnose food allergy.

Evidence:
- Page 21, Lines 1‑39 (Source 1)

Question: What gastrointestinal symptoms may be associated with food allergy?
Answer:
Gastrointestinal symptoms that may be linked to food allergy include:
- Stomach ache, nausea, vomiting and diarrhoea  
- Failing to thrive or faltering growth  
- Protein‑losing enteropathy  
- Blood in the stool  
- Gastro‑oesophageal reflux disease  
-

### Recommendation Generation

In this step, the system generates a recommendation based only on the retrieved evidence from the document. The recommendation must not include information from outside the provided context and should include the source page and line range

Answer the user's question using only the provided context.

After the answer, provide a recommendation only if a relevant recommendation is explicitly stated in the context.

Do not create or infer medical recommendations.

If no recommendation is available in the context, say:
"No specific recommendation was found in the provided document."

For the recommendation, include the source page and line range.

In [ ]:
# def build_recommendation_prompt(question, retrieved_chunks):

#     context_parts = []

#     for i, chunk in enumerate(retrieved_chunks, start=1):
#         context_parts.append(
#             f"""Source {i}
# Page: {chunk['page']}
# Lines: {chunk['start_line']} - {chunk['end_line']}

# {chunk['text']}"""
#         )

#     context = "\n\n".join(context_parts)

#     prompt = f"""
# You are a helpful assistant answering questions about food allergy.

# Answer the user's question using only the provided context.

# Rules:
# 1. Do not use outside knowledge.
# 2. Do not make up information.
# 3. Use only information explicitly stated in the context.
# 4. Do not infer or create medical recommendations.
# 5. Provide a recommendation only if it is explicitly supported by the context.
# 6. If no recommendation is available in the context, say:
# "No specific recommendation was found in the provided document."
# 7. After the recommendation, provide the evidence used.
# 8. For each evidence item, include its page number and line range.

# Context:
# {context}

# Question:
# {question}

# Answer:
# """

#     return prompt


def build_recommendation_prompt(question, retrieved_chunks):

    context_parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        context_parts.append(
            f"""Source {i}
Page: {chunk['page']}
Lines: {chunk['start_line']} - {chunk['end_line']}

{chunk['text']}"""
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are a helpful assistant answering questions about food allergy.

Answer the user's question using only the provided context.

Rules:
1. Do not use outside knowledge.
2. Do not make up information.
3. Use only information explicitly stated in the context.
4. Do not infer or create medical recommendations.
5. Provide "My Recommendation💡" only if a relevant recommendation is explicitly supported by the context.
6. Do not present general medical advice as a recommendation.
7. If no recommendation is available in the context, say:
"No specific recommendation was found in the provided document."
8. After the answer, provide the evidence used.
9. For each evidence item, include its page number and line range.
10. Always format the response using exactly these sections:
Answer:
Evidence:
My Recommendation:
Recommendation Evidence:

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [ ]:
query = "What should be done if food allergy is suspected?"

retrieved_chunks = retrieve_chunks(query, k=5)

prompt = build_recommendation_prompt(
    query,
    retrieved_chunks
)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b:free",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

Answer:
If food allergy is suspected, a healthcare professional should first take an allergy‑focused clinical history using the questions outlined in recommendation 1.1.3. Based on that history, the child or young person should be physically examined, paying particular attention to growth, signs of malnutrition, and allergy‑related comorbidities such as atopic eczema, asthma, and allergic rhinitis. If any of the referral criteria are met—such as faltering growth with gastrointestinal symptoms, lack of response to a single‑allergen elimination diet, acute systemic or severe delayed reactions, confirmed IgE‑mediated allergy with concurrent asthma, significant atopic eczema with suspected multiple or cross‑reactive food allergies, persisting parental suspicion despite a lack of supporting history, strong clinical suspicion with negative test results, or clinical suspicion of multiple food allergies—the child or young person should be referred to secondary or specialist care.

Evidence:
- 

In [ ]:
response = client.chat.completions.create(
    # model="openai/gpt-oss-20b:free",
    model="openrouter/free",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

answer = response.choices[0].message.content

print(answer)

Answer: If food allergy is suspected, the following steps should be taken:  
1. A healthcare professional with appropriate competencies (e.g., a GP) should take an allergy-focused clinical history using the questions outlined in recommendation 1.1.3.  
2. The child or young person should be physically examined, with particular attention to growth, signs of malnutrition, and allergy-related comorbidities such as atopic eczema, asthma, and allergic rhinitis.  
3. Referral to secondary or specialist care should be considered if any of the following apply:  
   - Faltering growth with gastrointestinal symptoms.  
   - Acute systemic reactions or severe delayed reactions.  
   - Significant atopic eczema with suspected multiple or cross-reactive food allergies.  
   - Persisting parental suspicion of food allergy despite a lack of supporting history.  
4. Diagnostic tests such as skin prick tests may be used, but only by healthcare professionals with appropriate competencies and in settings

### Final RAG Pipeline

The final RAG pipeline combines retrieval, context preparation, prompt construction, and language model generation. The system retrieves relevant document chunks and generates an evidence-based answer. Recommendation generation is applied only when the user's question explicitly asks for a recommendation or what should be done.


In [ ]:
def rag_pipeline(query):

    # Retrieve relevant chunks
    retrieved_chunks = retrieve_chunks(query, k=5)

    # Choose the appropriate prompt
    recommendation_keywords = [
        "what should be done",
        "what should I do",
        "recommendation",
        "recommend",
        "when should",
        "should be referred",
        "management"
    ]

    is_recommendation_question = any(
        keyword in query.lower()
        for keyword in recommendation_keywords
    )

    if is_recommendation_question:
        prompt = build_recommendation_prompt(
            query,
            retrieved_chunks
        )
    else:
        prompt = build_prompt(
            query,
            retrieved_chunks
        )

    # Generate answer
    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [ ]:
query = "What are the symptoms of food allergy?"

answer = rag_pipeline(query)

print(answer)

Answer:
The symptoms of food allergy include cutaneous symptoms (eruption, itching, rash, swelling), nasal symptoms (sneezing, itching, secretion, blockage), ocular symptoms (redness, itching, secretion), bronchial symptoms (cough, wheezing, shortness of breath), gastrointestinal symptoms (stomach ache, nausea, vomiting, diarrhoea), laryngeal symptoms (difficulty swallowing or speaking), and cardiovascular symptoms (palpitations, tachycardia, hypotension).

Evidence:
- Page 21, Lines 1-39


In [ ]:
query = "What should be done if food allergy is suspected?"

answer = rag_pipeline(query)

print(answer)

Answer:
If food allergy is suspected, take an allergy‑focused clinical history, physically examine the child for growth and signs of malnutrition and for allergy‑related comorbidities such as atopic eczema, asthma and allergic rhinitis, and consider referral to secondary or specialist care when any of the specified criteria are met (e.g., faltering growth with gastrointestinal symptoms, acute systemic or severe delayed reactions, significant atopic eczema with suspected multiple allergies, persisting parental suspicion despite lack of supporting history).

Evidence:
- Source 1: Page 13, Lines 27-41
- Source 2: Page 70, Lines 1-26
- Source 4: Page 14, Lines 29-52
- Source 5: Page 27, Lines 1-30

My Recommendation:
Consider referral to secondary or specialist care if any of the listed criteria are present.

Recommendation Evidence:
- Source 1: Page 13, Lines 27-41
- Source 2: Page 70, Lines 1-26


In [ ]:
# def rag_pipeline(query):

#     # 1. Retrieve relevant chunks
#     retrieved_chunks = retrieve_chunks(query, k=5)

#     # 2. Build the prompt
#     prompt = build_recommendation_prompt(
#         query,
#         retrieved_chunks
#     )

#     # 3. Generate the answer
#     response = client.chat.completions.create(
#         model="openrouter/free",
#         messages=[
#             {
#                 "role": "user",
#                 "content": prompt
#             }
#         ],
#         temperature=0
#     )

#     # 4. Return the final response
#     return response.choices[0].message.content

In [ ]:
# query = "What are the symptoms of food allergy?"

# answer = rag_pipeline(query)

# print(answer)

User Question
      ↓
Retrieve Top-K Chunks
      ↓
Prepare Context
      ↓
Detect Question Type
      ↓
 ┌───────────────┐
 │               │
Normal       Recommendation
Question        Question
 │               │
 ↓               ↓
RAG Prompt   Recommendation Prompt
 │               │
 └───────┬───────┘
         ↓
     OpenRouter
         ↓
       Answer
         ↓
      Evidence
         ↓
   My Recommendation